# Logistic Regression

## Load Dataset from sklearn

In [ ]:
!pip install pandas seaborn scikit-learn # 코드 실행


In [ ]:
import pandas as pd # 라이브러리 임포트
import seaborn as sns # 라이브러리 임포트
import matplotlib.pyplot as plt # 라이브러리 임포트

In [ ]:
from sklearn.datasets import load_breast_cancer # 라이브러리 임포트
cancer = load_breast_cancer() # 데이터셋 로드

print(cancer.DESCR) # 결과 출력

In [ ]:
df = pd.DataFrame(cancer.data, columns=cancer.feature_names) # 데이터프레임 생성
df['class'] = cancer.target # 변수 설정

df.tail() # 코드 실행

### Pair plot with mean features

In [ ]:
#sns.pairplot(df[['class'] + list(df.columns[:10])])
sns.pairplot(df[['class'] + list(df.columns[:10])], hue='class', palette='husl') # 변수 설정
plt.show() # 그래프 출력

### Pair plot with std features

In [ ]:
#sns.pairplot(df[['class'] + list(df.columns[10:20])])
sns.pairplot(df[['class'] + list(df.columns[10:20])], hue='class', palette='gray') # 변수 설정
plt.show() # 그래프 출력

### Pair plot with worst features

In [ ]:
#sns.pairplot(df[['class'] + list(df.columns[20:30])])
sns.pairplot(df[['class'] + list(df.columns[20:30])], hue='class', palette='coolwarm') # 변수 설정
plt.show() # 그래프 출력

### Select features

In [ ]:
cols = ["mean radius", "mean texture", # 변수 설정
        "mean smoothness", "mean compactness", "mean concave points", # 코드 실행
        "worst radius", "worst texture", # 코드 실행
        "worst smoothness", "worst compactness", "worst concave points", # 코드 실행
        "class"] # 코드 실행

In [ ]:
for c in cols[:-1]: # 반복문 수행
    sns.histplot(df, x=c, hue=cols[-1], bins=50, stat='probability') # 변수 설정
    plt.show() # 그래프 출력

## Train Model with PyTorch

In [ ]:
import torch # 라이브러리 임포트
import torch.nn as nn # 라이브러리 임포트
import torch.nn.functional as F # 라이브러리 임포트
import torch.optim as optim # 라이브러리 임포트

In [ ]:
data = torch.from_numpy(df[cols].values).float() # 텐서 변환

data.shape # 코드 실행

In [ ]:
# Split x and y.
x = data[:, :-1] # 변수 설정
y = data[:, -1:] # 변수 설정

print(x.shape, y.shape) # 결과 출력

In [ ]:
# Define configurations.
n_epochs = 200000 # 에포크 설정
learning_rate = 1e-2 # 학습률 설정
print_interval = 10000 # 변수 설정

In [ ]:
# Define costum model.
class MyModel(nn.Module): # 클래스 정의
    
    def __init__(self, input_dim, output_dim): # 함수 정의
        self.input_dim = input_dim # 변수 설정
        self.output_dim = output_dim # 변수 설정
        
        super().__init__() # 코드 실행
        
        self.linear = nn.Linear(input_dim, output_dim) # 선형층 정의
        self.act = nn.Sigmoid() # Sigmoid 활성화 함수
        
    def forward(self, x): # 함수 정의
        # |x| = (batch_size, input_dim)
        y = self.act(self.linear(x)) # 변수 설정
        # |y| = (batch_size, output_dim)
        
        return y # 결과 반환

In [ ]:
model = MyModel(input_dim=x.size(-1), # 변수 설정
                output_dim=y.size(-1)) # 변수 설정
crit = nn.BCELoss() # 손실 함수 정의

optimizer = optim.SGD(model.parameters(), # 최적화 알고리즘 설정
                      lr=learning_rate) # 학습률 설정

In [ ]:
for i in range(n_epochs): # 반복문 수행
    y_hat = model(x) # 모델 예측
    loss = crit(y_hat, y) # 변수 설정
    
    optimizer.zero_grad() # 기울기 초기화
    loss.backward() # 역전파 수행
    
    optimizer.step() # 가중치 업데이트
    
    if (i + 1) % print_interval == 0: # 조건문 수행
        print('Epoch %d: loss=%.4e' % (i + 1, loss)) # 결과 출력

## Let's see the result!

In [ ]:
correct_cnt = (y == (y_hat > .5)).sum() # 변수 설정
total_cnt = float(y.size(0)) # 변수 설정

print('Accuracy: %.4f' % (correct_cnt / total_cnt)) # 결과 출력

In [ ]:
df = pd.DataFrame(torch.cat([y, y_hat], dim=1).detach().numpy(), # 데이터프레임 생성
                  columns=["y", "y_hat"]) # 변수 설정

sns.histplot(df, x='y_hat', hue='y', bins=50, stat='probability') # 변수 설정
plt.show() # 그래프 출력